# SICD Orthorectification — Slant Range to Geographic Grid Projection

**Objective**: Project a SICD complex SAR image from slant-range geometry to a regular geographic grid (UTM or WGS-84) with optional DEM terrain correction.

## Overview

**Orthorectification** transforms imagery from sensor-native geometry (slant range, row/col pixels) to a **ground-referenced grid** (lat/lon or projected coordinates):
1. Define an **output grid** (UTM, WebMercator, or ENU)
2. For each output pixel, compute the corresponding **slant-range pixel** via geolocation inverse
3. **Resample** input data at those fractional pixel coordinates
4. Apply **DEM terrain correction** (optional) to map pixels to true ground height

## Theory

### Slant Range vs Ground Range

SAR imagery is natively in **slant range** — the line-of-sight distance from sensor to ground:
$$R_{slant} = \sqrt{R_{ground}^2 + h^2}$$
where $h$ is platform altitude.

Orthorectification reverses this: given a ground point $(lat, lon, h_{DEM})$, solve for the slant-range pixel $(row, col)$.

### DEM Terrain Correction

Without DEM:
- Ground points assumed at **ellipsoid height** (h = 0)
- Produces **layover** and **foreshortening** artifacts in mountainous terrain

With DEM:
- Each ground point uses **DEM-queried elevation** $h_{DEM}(lat, lon)$
- SICD R/Rdot inverse iteratively refines position using $h_{DEM}$
- Corrects geometric distortions from terrain relief

### Output Grids

GRDL supports three output projections:

| Grid Type | Coordinate System | Use Case |
|-----------|------------------|----------|
| **UTMGrid** | UTM zone (meters) | Local-scale analysis, preserves distances |
| **WebMercatorGrid** | EPSG:3857 (web maps) | Overlay on slippy maps (Google/OSM) |
| **ENUGrid** | East-North-Up (meters) | Radar-centric, local Cartesian |

### Modules Used

| Module | Purpose |
|--------|--------|
| `grdl.IO` | SICD reader via factory (`get_reader('sicd', ...)`) |
| `grdl.geolocation` | `create_geolocation()` auto-detect, DEM attachment |
| `grdl.geolocation.elevation` | `open_elevation()` for DEM loading |
| `grdl.image_processing.ortho` | `orthorectify()`, `UTMGrid`, `ENUGrid` |
| `napari` | Interactive visualization |

## Preamble — Version Check

In [ ]:
import grdl
from packaging.version import parse as parse_version

required = "0.6.1"
current = grdl.__version__

if parse_version(current) < parse_version(required):
    raise RuntimeError(
        f"GRDL {required}+ required. Found {current}. "
        f"Update: pip install --upgrade grdl"
    )

print(f"✓ GRDL {current}")

## Imports

In [ ]:
from pathlib import Path

import numpy as np
import napari

from grdl.IO import get_reader
from grdl.geolocation import create_geolocation
from grdl.image_processing.ortho import orthorectify, UTMGrid, ENUGrid

## Configuration — User Paths

In [ ]:
# Path to SICD file
SICD_PATH = Path("/path/to/your/sicd_file.nitf")

# Optional: DEM path for terrain correction
# Set to None to use ellipsoid (h=0)
DEM_PATH = None  # e.g., Path("/path/to/fabdem/tiles") or Path("/path/to/srtm.tif")

# Validate path
assert SICD_PATH.exists(), f"SICD file not found: {SICD_PATH}"

print(f"✓ SICD file: {SICD_PATH.name}")
if DEM_PATH:
    print(f"✓ DEM: {DEM_PATH}")
else:
    print("⚠ No DEM — using ellipsoid height (h=0)")

## Step 1: Load SICD Metadata and Extract Chip

In [ ]:
with get_reader('sicd', SICD_PATH) as reader:
    meta = reader.metadata
    rows, cols = meta.image_data.num_rows, meta.image_data.num_cols
    
    print(f"Image size: {rows} × {cols}")
    print(f"Pixel type: {meta.image_data.pixel_type}")
    print(f"Collection mode: {meta.collection_info.radar_mode.mode_type}")
    
    # Extract center chip (4096×4096 or smaller)
    chip_size = min(4096, rows, cols)
    r_start = (rows - chip_size) // 2
    c_start = (cols - chip_size) // 2
    
    chip = reader.read(
        row_start=r_start, row_end=r_start + chip_size,
        col_start=c_start, col_end=c_start + chip_size,
    )

print(f"\nExtracted chip: {chip.shape}, dtype: {chip.dtype}")
print(f"Chip location: rows [{r_start}, {r_start + chip_size}), cols [{c_start}, {c_start + chip_size})")

## Step 2: Create Geolocation Object

**Critical**: For chips, wrap the geolocation with `ChipGeolocation` to handle row/col offset.

In [ ]:
from grdl.geolocation.chip import ChipGeolocation

with get_reader('sicd', SICD_PATH) as reader:
    # Create geolocation for full image
    geo_full = create_geolocation(reader)
    
    # Wrap with chip offset
    geo = ChipGeolocation(
        geolocation=geo_full,
        row_offset=r_start,
        col_offset=c_start,
    )

print(f"✓ Geolocation: {type(geo_full).__name__}")
print(f"  Chip offset: row={r_start}, col={c_start}")

## Step 3 (Optional): Attach DEM to Geolocation

**Per GRDL architecture**: The geolocation object owns the DEM, not the orthorectifier.

In [ ]:
if DEM_PATH:
    from grdl.geolocation.elevation import open_elevation
    
    # Attach DEM to the wrapped geolocation's inner geolocation
    geo.geolocation.elevation = open_elevation(DEM_PATH)
    print(f"✓ DEM attached: {type(geo.geolocation.elevation).__name__}")
else:
    print("⚠ No DEM attached — projection uses ellipsoid (h=0)")

## Step 4: Compute Magnitude in Slant Range

**Important**: Compute magnitude *before* resampling. Resampling complex data causes phase cancellation.

In [ ]:
magnitude = np.abs(chip)

# dB scale for display
magnitude_db = 20 * np.log10(magnitude + 1e-12)

print(f"Magnitude (dB) range: [{magnitude_db.min():.1f}, {magnitude_db.max():.1f}]")

## Step 5: Define Output Grid — UTM Projection

In [ ]:
with get_reader('sicd', SICD_PATH) as reader:
    meta = reader.metadata
    scp_lat = meta.geo_data.scp.llh.lat
    scp_lon = meta.geo_data.scp.llh.lon

# UTM grid centered on SCP
output_grid = UTMGrid.from_center(
    center_lat=scp_lat,
    center_lon=scp_lon,
    width_m=chip_size * 0.5,   # Approximate width in meters
    height_m=chip_size * 0.5,
    pixel_size_m=0.5,           # 0.5 m ground sample distance
)

print(f"UTM Grid:")
print(f"  Zone: {output_grid.utm_zone}{output_grid.hemisphere}")
print(f"  Size: {output_grid.shape}")
print(f"  Pixel size: {output_grid.pixel_size_m} m")

## Step 6: Orthorectify to UTM Grid

In [ ]:
print("Running orthorectification...")

ortho_result = orthorectify(
    data=magnitude,
    geolocation=geo,
    output_grid=output_grid,
    interpolation='bilinear',  # or 'nearest', 'bicubic'
)

ortho_magnitude = ortho_result.data

print(f"✓ Orthorectified: {ortho_magnitude.shape}")
print(f"  Valid pixels: {np.sum(~np.isnan(ortho_magnitude))} / {ortho_magnitude.size}")

## Step 7: Visualization — Slant vs Ground Geometry

In [ ]:
# dB-scale orthorectified magnitude
ortho_db = 20 * np.log10(ortho_magnitude + 1e-12)
vmax = np.nanmax(ortho_db)
vmin = vmax - 50.0  # 50 dB dynamic range

viewer = napari.Viewer(title="SICD Orthorectification")

# Layer 1: Original slant-range magnitude
viewer.add_image(
    magnitude_db,
    name="Slant Range (dB)",
    colormap="gray",
    contrast_limits=[magnitude_db.max() - 50, magnitude_db.max()],
)

# Layer 2: Orthorectified UTM magnitude
viewer.add_image(
    ortho_db,
    name="UTM Ortho (dB)",
    colormap="gray",
    contrast_limits=[vmin, vmax],
    visible=True,
)

print("✓ Napari viewer launched")
print("  Layer 1: Slant range geometry (original)")
print("  Layer 2: UTM geographic grid (orthorectified)")

## Physical Interpretation

### Slant Range Distortions

In **slant-range geometry** (Layer 1):
- **Foreshortening**: Slopes facing the radar appear compressed
- **Layover**: Tall objects lean toward the radar (tops appear before bases)
- **Shadow**: Areas behind tall objects receive no illumination

These artifacts are **geometric** — inherent to side-looking radar, not sensor errors.

### Orthorectified Correction

In **UTM grid** (Layer 2):
- Pixels represent **ground positions** at true geographic coordinates
- With DEM: Terrain relief corrected — features align with map coordinates
- Without DEM: Ellipsoid projection — layover/shadow remain but position is geocoded

### Invalid Pixels (NaN)

NaN pixels in ortho output occur when:
- Ground point is **outside image footprint** (edge of output grid)
- DEM has **no coverage** at that location
- Geometry is **unsolvable** (e.g., layover region with multiple solutions)

### UTM vs WebMercator vs ENU

- **UTM**: Preserves distances and angles within a zone (±500 km from central meridian)
- **WebMercator**: Compatible with web maps (Google/OSM) but distorts at high latitudes
- **ENU**: Radar-centric Cartesian (East-North-Up from SCP) — ideal for local analysis

## Summary

Successfully demonstrated:
- ✅ SICD slant-range image loading via IO factory
- ✅ Chip extraction with offset-aware geolocation (`ChipGeolocation`)
- ✅ DEM attachment to geolocation object (architecture compliance)
- ✅ UTM grid definition and orthorectification
- ✅ Magnitude computation before resampling (avoids phase cancellation)
- ✅ Side-by-side slant vs ground geometry visualization

### Key GRDL Patterns
- `get_reader('sicd', path)` for format-agnostic loading
- `create_geolocation(reader)` for auto-detection
- **DEM ownership**: `geo.elevation = open_elevation(dem_path)`
- `ChipGeolocation` wrapper for chip offsets
- `orthorectify(data, geolocation, output_grid)` for single-call projection

### Next Steps
- Try `ENUGrid` for radar-centric local projection
- Attach DEM for terrain-corrected output
- Increase pixel_size_m (1.0, 2.0) for faster processing
- Export to GeoTIFF: `GeoTIFFWriter.write(ortho_result.data, ...)`
- Compare with SIDD ortho (pre-projected product, next notebook)